In [11]:
import os  # reads settings from the environment

import pandas as pd  # pandas, always nicknamed "pd"
from dotenv import load_dotenv  # loads .env, same as loader.py
from sqlalchemy import URL, create_engine  # the connection pandas prefers

pd.set_option("display.max_columns", None)  # show all columns, never "..."

load_dotenv()

url = URL.create(
    "postgresql+psycopg",                   # Postgres, through psycopg
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),         # must be a number here, not text
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)                 # a reusable door into the database

In [12]:
query = "SELECT * FROM pd_features_build WHERE sample_set = 'train'"
df = pd.read_sql(query, engine)   # run the query, keep the result as a table
df.shape                          # (rows, columns)

(131366, 25)

In [ ]:
print(df.head(10).T.to_string())

In [ ]:
df.info()

In [ ]:
print(df.describe().T.to_string())

In [ ]:
for col in ["amortization_type", "prepayment_penalty_indicator",
            "interest_only_indicator", "property_type", "channel"]:
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
print(df["msa"].isna().mean())
print(df["msa"].nunique())
print(df["msa"].value_counts().head(10))

In [ ]:
print(df["postal_code"].nunique())
print(df["property_state"].nunique())

In [ ]:
print("--- check 3: LTV over 100 ---")
high_ltv = df[df["original_ltv"] > 100]
print(len(high_ltv))
print(high_ltv["harp_indicator"].value_counts())

print("--- check 4: rate spread range, and the odd 276/288 terms ---")
print(df["rate_spread"].describe())
print(df[df["loan_term"].isin([276, 288])]["harp_indicator"].value_counts())

print("--- check 5: credit scores at the very bottom ---")
print((df["credit_score"] == 300).sum())
print((df["credit_score"] < 500).sum())

In [14]:
missing = df.isna().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(3)}))

                      missing     pct
credit_score               83   0.063
first_time_homebuyer       21   0.016
msa                     24167  18.397
mi_percentage               1   0.001
number_of_units            25   0.019
original_cltv               6   0.005
original_dti_ratio       2592   1.973
original_ltv                5   0.004
property_type               1   0.001
postal_code                 1   0.001
number_of_borrowers        47   0.036


In [ ]:
fill_numbers = ["credit_score", "number_of_borrowers", "number_of_units",
                "original_cltv", "original_ltv", "mi_percentage"]
print(df[fill_numbers].median())
print(df["first_time_homebuyer"].mode())
print(df["property_type"].mode())

print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))


In [13]:
df["year"] = "20" + df["loan_sequence_number"].str[1:3]
print(df.groupby("year")["target"].agg(["count", "mean"]))

      count      mean
year                 
2005  28440  0.010443
2006  27479  0.019797
2007  27691  0.050269
2008  29219  0.040966
2012  18537  0.001403


In [15]:
num_cols = ["credit_score", "original_ltv", "original_cltv", "original_dti_ratio",
            "original_upb", "rate_spread", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units"]
print(df.groupby("year")[num_cols].mean().round(2).T)

year                      2005       2006       2007       2008       2012
credit_score            724.96     723.43     724.52     742.11     766.58
original_ltv             68.88      69.81      71.41      69.90      67.90
original_cltv            70.51      72.30      74.05      71.09      68.92
original_dti_ratio       35.22      36.45      36.67      36.17      30.90
original_upb         169357.35  178280.98  183080.10  204523.60  222187.19
rate_spread               0.07       0.02       0.06       0.01       0.14
loan_term               322.06     334.87     339.59     334.69     289.88
mi_percentage             3.00       3.24       4.94       4.26       3.03
number_of_borrowers       1.59       1.55       1.53       1.52       1.62
number_of_units           1.03       1.03       1.03       1.03       1.02


In [ ]:
for col in ["loan_purpose", "channel", "occupancy_status", "property_type",
            "first_time_homebuyer", "harp_indicator"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

In [ ]:
for col in ["property_type", "first_time_homebuyer"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

y2012 = df[df["year"] == "2012"]
print(y2012.groupby("harp_indicator")[["target", "rate_spread"]].mean())

In [16]:
print((df["original_ltv"] > 100).sum())

1


In [17]:
print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))

                     count      mean
original_dti_ratio                  
False               128774  0.026209
True                  2592  0.031250
